# UAS Praktikum Machine Learning — End-to-End Deep Learning Pipeline
## AI-Powered Adaptive Learning Recommender

**Goal:** Membangun sistem klasifikasi `course_completed` (Lulus / Tidak Lulus) menggunakan data analytics pembelajaran digital.

| Komponen | Detail |
|---|---|
| **Dataset** | `digital_learning_analytics_100k.csv` (100.000 baris, 43 kolom) |
| **Target** | `course_completed` — Binary Classification |
| **Preprocessing** | Missing value imputation, Outlier clipping (IQR), Encoding (Ordinal + One-Hot), Scaling |
| **Dimensionality Reduction** | PCA, LDA, PCA+LDA Hybrid, Autoencoder → dipilih terbaik |
| **Model Deep Learning** | Deep MLP (TensorFlow/Keras) — model terbaik & tercepat |
| **Model PyTorch** | RNN — model terbaik PyTorch |
| **Imbalance Handling** | SMOTETomek |
| **Regularisasi** | BatchNorm, Dropout, L2, EarlyStopping, ReduceLROnPlateau |
| **Output** | Best model (joblib + .keras + .pt), visualisasi, HuggingFace deploy plan |

## 1. Import Semua Library

In [71]:
# =============================================
# 1. IMPORT SEMUA LIBRARY YANG DIBUTUHKAN
# =============================================

# --- Standard Library ---
import os, sys, warnings, time, json
import numpy as np
import pandas as pd

# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)

# --- Matplotlib (agar bisa save plot tanpa GUI) ---
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# --- Scikit-Learn: Preprocessing, DR, Splitting, Metrics ---
import sklearn, joblib, imblearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from imblearn.combine import SMOTETomek

# --- TensorFlow / Keras ---
import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers

# --- PyTorch ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --- Warnings off ---
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# --- Device info ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'TensorFlow  : {tf.__version__}')
print(f'PyTorch    : {torch.__version__}')
print(f'Device     : {DEVICE}')
print(f'Scikit-Learn : {sklearn.__version__}')
print(f'Imbalanced-Learn : {imblearn.__version__}')
print(f'\nSemua library berhasil di-import!')

TensorFlow  : 2.19.0
PyTorch    : 2.10.0+cu128
Device     : cuda
Scikit-Learn : 1.6.1
Imbalanced-Learn : 0.14.1

Semua library berhasil di-import!


## 2. Data Loading & Understanding

In [72]:
# =============================================
# 2. DATA LOADING & MEMAHAMI STRUKTUR DATASET
# =============================================

DATA_PATH = '/kaggle/input/datasets/misbahulmuttaqin/df-ml-prak-uas/digital_learning_analytics_100k.csv'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Load dataset penuh ---
df = pd.read_csv(DATA_PATH)
print(f'Dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'\n--- 5 Baris Pertama ---')
print(df.head().to_string())

print(f'\n--- Info Tipe Data ---')
print(df.info())

print(f'\n--- Statistik Deskriptif ---')
print(df.describe().T.to_string())

Dataset: 100,000 baris x 43 kolom

--- 5 Baris Pertama ---
    learner_id  age      gender           education_level        country   employment_status  prior_online_courses  digital_literacy_score       app_category  daily_app_minutes  session_count_weekly  app_completion_rate  in_app_quiz_score  gamification_engagement  skill_pre_score  skill_post_score essay_topic_category  essay_word_count  essay_grammar_errors  essay_vocabulary_richness  essay_coherence_score  human_grader_score  automated_score mooc_platform       course_category  course_duration_weeks  video_completion_pct  assignment_submission_rate  forum_posts  peer_review_given  course_completed learning_path_type  content_difficulty_avg  content_recommendations_followed  knowledge_gaps_identified  remediation_modules_completed  time_to_mastery_hours  mastery_score  learning_efficiency_score enrollment_date last_activity_date  total_learning_hours  engagement_consistency
0  LRN00000001   28  Non-binary                  Gradu

## 3. Exploratory Data Analysis (EDA)

In [73]:
# =============================================
# 3. EDA: Missing Values, Target Distribution, Outliers
# =============================================

# --- 3a. Missing Values ---
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct%': missing_pct})
missing_only = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)
print(missing_only if len(missing_only) > 0 else 'Tidak ada missing values!')

# --- 3b. Target Distribution ---
print('\n=== TARGET DISTRIBUTION (course_completed) ===')
print(df['course_completed'].value_counts())
print(f'\nImbalance ratio: {df["course_completed"].value_counts().max() / df["course_completed"].value_counts().min():.2f} : 1')

# --- 3c. Visualisasi Target ---
fig, ax = plt.subplots(figsize=(6, 4))
df['course_completed'].value_counts().plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], alpha=0.85)
ax.set_title('Target Distribution: course_completed')
ax.set_ylabel('Count')
ax.set_xticklabels(['Not Completed (0)', 'Completed (1)'], rotation=0)
for i, v in enumerate(df['course_completed'].value_counts().values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_target_distribution.png', dpi=150)
plt.show()
print('Plot disimpan.')

=== MISSING VALUES ===
                                  count  pct%
peer_review_given                  4153  4.15
forum_posts                        2967  2.97
gamification_engagement            2490  2.49
essay_coherence_score              2022  2.02
essay_vocabulary_richness          1973  1.97
content_recommendations_followed   1514  1.51
learning_efficiency_score           973  0.97

=== TARGET DISTRIBUTION (course_completed) ===
course_completed
False    69360
True     30640
Name: count, dtype: int64

Imbalance ratio: 2.26 : 1
Plot disimpan.


## 4. Data Cleaning

In [74]:
# =============================================
# 4. DATA CLEANING: Drop kolom, Duplikat, Outliers
# =============================================

# --- 4a. Drop kolom yang tidak relevan / data leakage ---
DROP_COLS = [
    'learner_id',            # ID unik, tidak informatif
    'enrollment_date',       # Tanggal, tidak predictif
    'last_activity_date',    # Tanggal, tidak predictif
    'human_grader_score',    # Data leakage (terlalu berkorelasi dengan target)
    'automated_score',       # Data leakage
]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')
print(f'Dropped {len(DROP_COLS)} kolom (ID, tanggal, potential leakage)')

# --- 4b. Handle Duplicates ---
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f'Duplicates dihapus: {n_dup}')
print(f'Shape setelah cleaning: {df.shape}')

# --- 4c. Target sebagai int ---
TARGET_COL = 'course_completed'
df[TARGET_COL] = df[TARGET_COL].astype(int)

# --- 4d. Pisahkan kolom numerik vs kategorikal ---
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET_COL]
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

ordinal_map = {
    'education_level': ['High School', 'Some College', "Bachelor's", 'Graduate', 'Doctoral'],
    'learning_path_type': ['Linear', 'Branched', 'Adaptive'],
}
ordinal_cols = [c for c in ordinal_map if c in cat_cols]
nominal_cols = [c for c in cat_cols if c not in ordinal_cols]

print(f'\nFitur numerik  : {len(num_cols)} kolom')
print(f'Fitur ordinal  : {ordinal_cols}')
print(f'Fitur nominal  : {len(nominal_cols)} kolom')

Dropped 5 kolom (ID, tanggal, potential leakage)
Duplicates dihapus: 0
Shape setelah cleaning: (100000, 38)

Fitur numerik  : 28 kolom
Fitur ordinal  : ['education_level', 'learning_path_type']
Fitur nominal  : 7 kolom


In [75]:
# =============================================
# 4e. HANDLE OUTLIERS (IQR Clipping)
# =============================================

print('=== OUTLIER HANDLING (IQR Clipping) ===')
outlier_info = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    if n_out > 0:
        outlier_info[col] = n_out
        df[col] = df[col].clip(lower=lower, upper=upper)

if outlier_info:
    oi = pd.DataFrame.from_dict(outlier_info, orient='index', columns=['outliers'])
    oi = oi.sort_values('outliers', ascending=False)
    print(oi.to_string())
    print(f'\nTotal kolom dengan outlier: {len(outlier_info)}')
else:
    print('Tidak ada outlier terdeteksi.')

=== OUTLIER HANDLING (IQR Clipping) ===
                               outliers
total_learning_hours               8474
remediation_modules_completed      6727
prior_online_courses               5985
learning_efficiency_score          4977
engagement_consistency             3388
forum_posts                        2602
knowledge_gaps_identified          2175
session_count_weekly               1626
daily_app_minutes                  1624
peer_review_given                  1592
essay_vocabulary_richness          1481
essay_word_count                   1360
digital_literacy_score             1000
essay_grammar_errors                974
content_difficulty_avg              846
mastery_score                       759
skill_post_score                    666
time_to_mastery_hours               485
in_app_quiz_score                   363
assignment_submission_rate          285
video_completion_pct                132
essay_coherence_score               108

Total kolom dengan outlier: 22


## 5. Feature Transformation Pipeline

In [76]:
# =============================================
# 5. FEATURE TRANSFORMATION: Encoding + Imputation + Scaling
# =============================================

# --- 5a. Pipeline untuk fitur numerik: Median Impute → StandardScaler ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# --- 5b. Pipeline untuk fitur ordinal: Mode Impute → OrdinalEncoder → StandardScaler ---
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[ordinal_map[c] for c in ordinal_cols],
        handle_unknown='use_encoded_value', unknown_value=-1
    )),
    ('scaler', StandardScaler()),
])

# --- 5c. Pipeline untuk fitur nominal: Mode Impute → OneHotEncoder ---
nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# --- 5d. Gabungkan semua pipeline dengan ColumnTransformer ---
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('ord', ordinal_pipeline, ordinal_cols),
    ('nom', nominal_pipeline, nominal_cols),
], remainder='drop')

print('Preprocessor pipeline dibuat:')
print(f'  - Numerik ({len(num_cols)} fitur): Median Imputation → StandardScaler')
print(f'  - Ordinal ({len(ordinal_cols)} fitur): Mode Impute → OrdinalEncoder → StandardScaler')
print(f'  - Nominal ({len(nominal_cols)} fitur): Mode Impute → OneHotEncoder')

Preprocessor pipeline dibuat:
  - Numerik (28 fitur): Median Imputation → StandardScaler
  - Ordinal (2 fitur): Mode Impute → OrdinalEncoder → StandardScaler
  - Nominal (7 fitur): Mode Impute → OneHotEncoder


## 6. Data Splitting (80:10:10)

In [77]:
# =============================================
# 6. DATA SPLITTING: Train 80% | Val 10% | Test 10%
# =============================================

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values

# Split pertama: 80% train, 20% sisa
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
# Split kedua: 50% val, 50% test (dari 20% → 10% + 10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp
)

print(f'Train: {len(y_train):,} samples ({np.sum(y_train==0)} Class 0, {np.sum(y_train==1)} Class 1)')
print(f'Val  : {len(y_val):,} samples')
print(f'Test : {len(y_test):,} samples')

# --- Apply preprocessing ---
print('\nMenerapkan preprocessing pipeline...')
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

n_features = X_train_proc.shape[1]
print(f'Fitur setelah encoding: {n_features}')
print(f'Shape: Train {X_train_proc.shape}, Val {X_val_proc.shape}, Test {X_test_proc.shape}')

Train: 80,000 samples (55488 Class 0, 24512 Class 1)
Val  : 10,000 samples
Test : 10,000 samples

Menerapkan preprocessing pipeline...
Fitur setelah encoding: 116
Shape: Train (80000, 116), Val (10000, 116), Test (10000, 116)


## 7. Dimensionality Reduction (PCA, LDA, Autoencoder)

In [78]:
# =============================================
# 7a. PCA (Principal Component Analysis) — 95% varians
# =============================================

pca_full = PCA(random_state=SEED).fit(X_train_proc)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_pca = max(int(np.argmax(cumvar >= 0.95)) + 1, 10)

pca = PCA(n_components=n_pca, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_proc)
X_val_pca   = pca.transform(X_val_proc)
X_test_pca  = pca.transform(X_test_proc)

print(f'PCA: {n_features} → {n_pca} komponen ({sum(pca.explained_variance_ratio_)*100:.2f}% varians)')

# --- Plot variance ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, min(n_features+1, 80)), cumvar[:min(n_features, 79)], 'b-o', markersize=3)
ax.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
ax.axvline(x=n_pca, color='g', linestyle='--', alpha=0.7, label=f'Selected: {n_pca} PCs')
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA — Explained Variance Analysis')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pca_variance_analysis.png', dpi=150)
plt.show()

PCA: 116 → 49 komponen (95.18% varians)


In [79]:
# =============================================
# 7b. LDA (Linear Discriminant Analysis)
# =============================================

# LDA untuk binary classification → 1 komponen (max n_classes - 1)
lda = LDA(n_components=1)
X_train_lda = lda.fit_transform(X_train_proc, y_train)
X_val_lda   = lda.transform(X_val_proc)
X_test_lda  = lda.transform(X_test_proc)
print(f'LDA: {n_features} → 1 komponen (class-separability)')

# --- 7c. PCA + LDA Hybrid (gabungkan keduanya) ---
X_train_pca_lda = np.hstack([X_train_pca, X_train_lda])
X_val_pca_lda   = np.hstack([X_val_pca, X_val_lda])
X_test_pca_lda  = np.hstack([X_test_pca, X_test_lda])
print(f'PCA+LDA Hybrid: {n_pca} + 1 = {n_pca + 1} komponen')

LDA: 116 → 1 komponen (class-separability)
PCA+LDA Hybrid: 49 + 1 = 50 komponen


In [80]:
# =============================================
# 7d. AUTOENCODER (Learned Compression via PyTorch)
# =============================================

AE_LATENT = min(25, n_features // 3)

class Autoencoder(nn.Module):
    """Autoencoder untuk dimensionality reduction."""
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),   nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, latent_dim), nn.BatchNorm1d(latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

# --- Training Autoencoder ---
ae_model = Autoencoder(n_features, AE_LATENT).to(DEVICE)
ae_opt = optim.Adam(ae_model.parameters(), lr=1e-3, weight_decay=1e-5)
ae_crit = nn.MSELoss()

X_train_t = torch.FloatTensor(X_train_proc).to(DEVICE)
ae_loader = DataLoader(TensorDataset(X_train_t), batch_size=256, shuffle=True)

print(f'Autoencoder: {n_features} → {AE_LATENT} latent dim')
print('Training Autoencoder...')
ae_model.train()
for epoch in range(50):
    total_loss = 0
    for (xb,) in ae_loader:
        ae_opt.zero_grad()
        loss = ae_crit(ae_model(xb), xb)
        loss.backward()
        ae_opt.step()
        total_loss += loss.item() * len(xb)
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d} | Recon Loss: {total_loss / len(X_train_t):.4f}')

# --- Ekstrak fitur latent ---
ae_model.eval()
with torch.no_grad():
    X_train_ae = ae_model.encoder(X_train_t).cpu().numpy()
    X_val_ae   = ae_model.encoder(torch.FloatTensor(X_val_proc).to(DEVICE)).cpu().numpy()
    X_test_ae  = ae_model.encoder(torch.FloatTensor(X_test_proc).to(DEVICE)).cpu().numpy()
print(f'Autoencoder selesai. Shape: {X_train_ae.shape}')

Autoencoder: 116 → 25 latent dim
Training Autoencoder...
  Epoch  10 | Recon Loss: 0.0823
  Epoch  20 | Recon Loss: 0.0765
  Epoch  30 | Recon Loss: 0.0722
  Epoch  40 | Recon Loss: 0.0703
  Epoch  50 | Recon Loss: 0.0690
Autoencoder selesai. Shape: (80000, 25)


In [81]:
# =============================================
# 7e. BENCHMARK TEKNIK DR — Pilih Terbaik
# =============================================

from sklearn.linear_model import LogisticRegression

dr_datasets = {
    'PCA':         (X_train_pca,     X_val_pca,     X_test_pca),
    'PCA+LDA':     (X_train_pca_lda, X_val_pca_lda, X_test_pca_lda),
    'Autoencoder': (X_train_ae,      X_val_ae,      X_test_ae),
}

dr_scores = {}
for name, (Xtr, Xva, Xte) in dr_datasets.items():
    lr = LogisticRegression(max_iter=500, random_state=SEED)
    lr.fit(Xtr, y_train)
    preds = lr.predict(Xte)
    dr_scores[name] = f1_score(y_test, preds, zero_division=0)
    print(f'  {name:12s}: F1 = {dr_scores[name]:.4f}  (dim={Xtr.shape[1]})')

BEST_DR = max(dr_scores, key=dr_scores.get)
print(f'\n>>> Teknik DR terbaik: {BEST_DR} (F1={dr_scores[BEST_DR]:.4f})')

# --- Pilih dataset DR terbaik ---
if BEST_DR == 'PCA':
    X_train_dr, X_val_dr, X_test_dr = X_train_pca, X_val_pca, X_test_pca
elif BEST_DR == 'PCA+LDA':
    X_train_dr, X_val_dr, X_test_dr = X_train_pca_lda, X_val_pca_lda, X_test_pca_lda
else:
    X_train_dr, X_val_dr, X_test_dr = X_train_ae, X_val_ae, X_test_ae

INPUT_DIM = X_train_dr.shape[1]
print(f'Menggunakan {BEST_DR} dengan {INPUT_DIM} dimensi.')

# --- Plot DR comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Bar chart F1
names = list(dr_scores.keys())
vals = list(dr_scores.values())
axes[0].bar(names, vals, color=['steelblue','darkorange','green'], alpha=0.85)
for i, v in enumerate(vals):
    axes[0].text(i, v+0.005, f'{v:.3f}', ha='center', fontweight='bold')
axes[0].set_title('DR Technique — F1-Score Benchmark')
axes[0].set_ylabel('F1-Score')
axes[0].set_ylim(0, 1.0)
# Bar chart dims
dims = [X_train_pca.shape[1], X_train_pca_lda.shape[1], X_train_ae.shape[1]]
axes[1].bar(names, dims, color=['steelblue','darkorange','green'], alpha=0.85)
for i, d in enumerate(dims):
    axes[1].text(i, d+0.5, str(d), ha='center', fontweight='bold')
axes[1].set_title('DR Technique — Feature Count')
axes[1].set_ylabel('Number of Components')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dim_reduction_comparison.png', dpi=150)
plt.show()

  PCA         : F1 = 0.8135  (dim=49)
  PCA+LDA     : F1 = 0.8137  (dim=50)
  Autoencoder : F1 = 0.7802  (dim=25)

>>> Teknik DR terbaik: PCA+LDA (F1=0.8137)
Menggunakan PCA+LDA dengan 50 dimensi.


## 8. Handle Imbalanced Data (SMOTETomek)

In [82]:
# =============================================
# 8. HANDLE IMBALANCED DATA MENGGUNAKAN SMOTETOMEK
# =============================================

print(f'Sebelum SMOTE — Train: Class 0={np.sum(y_train==0):,}, Class 1={np.sum(y_train==1):,}')

smote = SMOTETomek(random_state=SEED)
X_train_res, y_train_res = smote.fit_resample(X_train_dr, y_train)

print(f'Sesudah SMOTE — Train: Class 0={np.sum(y_train_res==0):,}, Class 1={np.sum(y_train_res==1):,}')
print(f'Total train samples setelah resampling: {X_train_res.shape[0]:,}')

Sebelum SMOTE — Train: Class 0=55,488, Class 1=24,512
Sesudah SMOTE — Train: Class 0=55,268, Class 1=55,268
Total train samples setelah resampling: 110,536


## 9. Model TensorFlow/Keras — Deep MLP (Model Terbaik & Tercepat)

In [83]:
# =============================================
# 9. MODEL 1: DEEP MLP (TensorFlow/Keras)
# Arsitektur terbaik dari eksperimen sebelumnya
# 3 Hidden Layers + BatchNorm + Dropout + L2
# =============================================

# --- Callbacks: Early Stopping + Reduce LR ---
early_stop_tf = callbacks.EarlyStopping(
    monitor='val_loss', patience=12, restore_best_weights=True, verbose=1
)
reduce_lr_tf = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6, verbose=1
)

# --- Class Weights ---
cc = np.bincount(y_train_res)
total_cc = cc.sum()
class_weight_tf = {0: total_cc / (2 * cc[0]), 1: total_cc / (2 * cc[1])}
print(f'Class weights: {class_weight_tf}')

# --- Arsitektur Deep MLP ---
model_tf = models.Sequential([
    layers.Input(shape=(INPUT_DIM,)),
    # Hidden Layer 1
    layers.Dense(128, kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.4),
    # Hidden Layer 2
    layers.Dense(64, kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    # Hidden Layer 3
    layers.Dense(32, kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    # Output Layer (Binary Classification)
    layers.Dense(1, activation='sigmoid'),
], name='DeepMLP')

model_tf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)

model_tf.summary()

Class weights: {0: np.float64(1.0), 1: np.float64(1.0)}


Model: "DeepMLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 128)            │         6,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,793 (69.50 KB)

 Trainable params: 17,345 (67.75 KB)

 Non-trainable params: 448 (1.75 KB)

In [84]:
# =============================================
# 9b. TRAINING Deep MLP
# =============================================

print('=== TRAINING Deep MLP (TensorFlow) ===')
t0 = time.time()

history_tf = model_tf.fit(
    X_train_res, y_train_res,
    validation_data=(X_val_dr, y_val),
    epochs=100,
    batch_size=256,    # batch besar untuk 100k data → training cepat
    class_weight=class_weight_tf,
    callbacks=[early_stop_tf, reduce_lr_tf],
    verbose=1,
)

tf_time = time.time() - t0
print(f'\nTraining selesai dalam {tf_time:.1f} detik ({tf_time/60:.1f} menit)')

=== TRAINING Deep MLP (TensorFlow) ===
Epoch 1/100
432/432 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.8447 - auc: 0.9224 - loss: 0.5034 - val_accuracy: 0.8751 - val_auc: 0.9403 - val_loss: 0.4165 - learning_rate: 0.0010
Epoch 2/100
432/432 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8784 - auc: 0.9479 - loss: 0.3631 - val_accuracy: 0.8812 - val_auc: 0.9513 - val_loss: 0.3397 - learning_rate: 0.0010
Epoch 3/100
432/432 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8887 - auc: 0.9565 - loss: 0.3055 - val_accuracy: 0.8928 - val_auc: 0.9607 - val_loss: 0.2960 - learning_rate: 0.0010
Epoch 4/100
432/432 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8967 - auc: 0.9623 - loss: 0.2765 - val_accuracy: 0.9084 - val_auc: 0.9662 - val_loss: 0.2615 - learning_rate: 0.0010
Epoch 5/100
432/432 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9028 - auc: 0.9661 - loss: 0.2603 - val_accuracy: 0.9096 - val_auc: 0.9674 - val_loss: 0.2496 - learning_rate: 0.0010
Epoch 6/100
432/432 ━━━━━━━━━━━━━━

In [85]:
# =============================================
# 9c. EVALUASI Deep MLP
# =============================================

y_prob_tf = model_tf.predict(X_test_dr, verbose=0).ravel()
y_pred_tf = (y_prob_tf >= 0.5).astype(int)

acc_tf  = accuracy_score(y_test, y_pred_tf)
prec_tf = precision_score(y_test, y_pred_tf, zero_division=0)
rec_tf  = recall_score(y_test, y_pred_tf, zero_division=0)
f1_tf   = f1_score(y_test, y_pred_tf, zero_division=0)
auc_tf  = roc_auc_score(y_test, y_prob_tf)
cm_tf   = confusion_matrix(y_test, y_pred_tf)

print('=== Deep MLP (TensorFlow) — EVALUASI ===')
print(f'  Accuracy : {acc_tf:.4f}')
print(f'  Precision: {prec_tf:.4f}')
print(f'  Recall   : {rec_tf:.4f}')
print(f'  F1-Score : {f1_tf:.4f}')
print(f'  AUC-ROC  : {auc_tf:.4f}')
print(f'  Waktu    : {tf_time:.1f}s')
print(f'\n  Confusion Matrix:\n    {cm_tf[0]}\n    {cm_tf[1]}')
print(f'\n{classification_report(y_test, y_pred_tf, target_names=["Not Completed","Completed"])}')

=== Deep MLP (TensorFlow) — EVALUASI ===
  Accuracy : 0.9268
  Precision: 0.8764
  Recall   : 0.8861
  F1-Score : 0.8812
  AUC-ROC  : 0.9768
  Waktu    : 143.1s

  Confusion Matrix:
    [6553  383]
    [ 349 2715]

               precision    recall  f1-score   support

Not Completed       0.95      0.94      0.95      6936
    Completed       0.88      0.89      0.88      3064

     accuracy                           0.93     10000
    macro avg       0.91      0.92      0.91     10000
 weighted avg       0.93      0.93      0.93     10000



## 10. Model PyTorch — RNN (Model Terbaik PyTorch)

In [86]:
# =============================================
# 10a. RESHAPE DATA UNTUK SEQUENTIAL MODEL (RNN)
# Data tabular → format (batch, seq_len, feat_per_step)
# =============================================

SEQ_LEN = 5
FEAT_PER_STEP = INPUT_DIM // SEQ_LEN

def reshape_to_seq(X, seq_len):
    trim = X.shape[1] - (seq_len * (X.shape[1] // seq_len))
    if trim > 0:
        X = X[:, :X.shape[1] - trim]
    return X.reshape(-1, seq_len, X.shape[1] // seq_len)
    
X_train_seq = reshape_to_seq(X_train_res, SEQ_LEN)
X_val_seq   = reshape_to_seq(X_val_dr, SEQ_LEN)
X_test_seq  = reshape_to_seq(X_test_dr, SEQ_LEN)
FEAT_PER_STEP = X_train_seq.shape[2]

print(f'Sequential format: (batch, {SEQ_LEN}, {FEAT_PER_STEP})')

# --- DataLoader ---
def make_loader(X, y, bs=256, shuffle=True):
    return DataLoader(TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y)),
                      batch_size=bs, shuffle=shuffle)

train_loader_pt = make_loader(X_train_seq, y_train_res, 256, True)
val_loader_pt   = make_loader(X_val_seq, y_val, 256, False)
test_loader_pt  = make_loader(X_test_seq, y_test, 256, False)

Sequential format: (batch, 5, 10)


In [87]:
# =============================================
# 10b. DEFINISI ARSITEKTUR RNN (PyTorch)
# =============================================

class RNNModel(nn.Module):
    """Vanilla RNN dengan 2 layer, BatchNorm, Dropout."""
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, dropout=0.3, nonlinearity='tanh')
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        out, _ = self.rnn(x)       # (batch, seq, hidden)
        out = out[:, -1, :]        # ambil timestep terakhir
        out = self.bn(out)
        out = self.dropout(out)
        return self.fc(out)

model_pt = RNNModel(FEAT_PER_STEP, hidden_dim=128, num_layers=2).to(DEVICE)
print(f'RNN Parameters: {sum(p.numel() for p in model_pt.parameters()):,}')

RNN Parameters: 61,569


In [88]:
# =============================================
# 10c. TRAINING RNN (PyTorch)
# =============================================

criterion_pt = nn.BCELoss()
optimizer_pt = optim.Adam(model_pt.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler_pt = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_pt, mode='min', factor=0.5, patience=6, min_lr=1e-6
)

EPOCHS = 100
PATIENCE = 12
history_pt = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
best_state_pt = None
wait = 0

print('=== TRAINING RNN (PyTorch) ===')
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model_pt.train()
    running_loss = 0
    for xb, yb in train_loader_pt:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_pt.zero_grad()
        out = model_pt(xb).squeeze(-1)
        loss = criterion_pt(out, yb)
        loss.backward()
        optimizer_pt.step()
        running_loss += loss.item() * len(xb)
    train_loss = running_loss / len(train_loader_pt.dataset)

    # --- Validate ---
    model_pt.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader_pt:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model_pt(xb).squeeze(-1)
            loss = criterion_pt(out, yb)
            val_loss += loss.item() * len(xb)
            preds = (out >= 0.5).float()
            correct += (preds == yb).sum().item()
            total += len(yb)
    val_loss /= len(val_loader_pt.dataset)
    val_acc = correct / total

    history_pt['train_loss'].append(train_loss)
    history_pt['val_loss'].append(val_loss)
    history_pt['val_acc'].append(val_acc)
    scheduler_pt.step(val_loss)

    # --- Early Stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_pt = {k: v.cpu().clone() for k, v in model_pt.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f'  Early stopping di epoch {epoch} (best val_loss={best_val_loss:.4f})')
            break

    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

# Restore best weights
if best_state_pt:
    model_pt.load_state_dict(best_state_pt)
model_pt.eval()

pt_time = time.time() - t0
print(f'\nTraining selesai dalam {pt_time:.1f} detik ({pt_time/60:.1f} menit)')

=== TRAINING RNN (PyTorch) ===
  Epoch  10 | Train Loss: 0.2148 | Val Loss: 0.2285 | Val Acc: 0.9056
  Epoch  20 | Train Loss: 0.1883 | Val Loss: 0.2300 | Val Acc: 0.9053
  Epoch  30 | Train Loss: 0.1592 | Val Loss: 0.2050 | Val Acc: 0.9141
  Early stopping di epoch 39 (best val_loss=0.1946)

Training selesai dalam 81.0 detik (1.4 menit)


In [89]:
# =============================================
# 10d. EVALUASI RNN (PyTorch)
# =============================================

all_probs, all_preds, all_true = [], [], []
with torch.no_grad():
    for xb, yb in test_loader_pt:
        out = model_pt(xb.to(DEVICE)).squeeze(-1).cpu().numpy().ravel()
        all_probs.extend(out)
        all_preds.extend((out >= 0.5).astype(int))
        all_true.extend(yb.numpy().ravel())

y_prob_pt = np.array(all_probs)
y_pred_pt = np.array(all_preds)

acc_pt  = accuracy_score(y_test, y_pred_pt)
prec_pt = precision_score(y_test, y_pred_pt, zero_division=0)
rec_pt  = recall_score(y_test, y_pred_pt, zero_division=0)
f1_pt   = f1_score(y_test, y_pred_pt, zero_division=0)
auc_pt  = roc_auc_score(y_test, y_prob_pt)
cm_pt   = confusion_matrix(y_test, y_pred_pt)

print('=== RNN (PyTorch) — EVALUASI ===')
print(f'  Accuracy : {acc_pt:.4f}')
print(f'  Precision: {prec_pt:.4f}')
print(f'  Recall   : {rec_pt:.4f}')
print(f'  F1-Score : {f1_pt:.4f}')
print(f'  AUC-ROC  : {auc_pt:.4f}')
print(f'  Waktu    : {pt_time:.1f}s')
print(f'\n  Confusion Matrix:\n    {cm_pt[0]}\n    {cm_pt[1]}')
print(f'\n{classification_report(y_test, y_pred_pt, target_names=["Not Completed","Completed"])}')

=== RNN (PyTorch) — EVALUASI ===
  Accuracy : 0.9204
  Precision: 0.8757
  Recall   : 0.8626
  F1-Score : 0.8691
  AUC-ROC  : 0.9731
  Waktu    : 81.0s

  Confusion Matrix:
    [6561  375]
    [ 421 2643]

               precision    recall  f1-score   support

Not Completed       0.94      0.95      0.94      6936
    Completed       0.88      0.86      0.87      3064

     accuracy                           0.92     10000
    macro avg       0.91      0.90      0.91     10000
 weighted avg       0.92      0.92      0.92     10000



## 11. Perbandingan Model & Visualisasi

In [90]:
# =============================================
# 11a. TABEL PERBANDINGAN DEEP MLP vs RNN
# =============================================

comparison = pd.DataFrame({
    'Metrik': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'Waktu Training (s)'],
    'Deep MLP (TensorFlow)': [acc_tf, prec_tf, rec_tf, f1_tf, auc_tf, round(tf_time, 1)],
    'RNN (PyTorch)':          [acc_pt, prec_pt, rec_pt, f1_pt, auc_pt, round(pt_time, 1)],
})
print(comparison.to_string(index=False))

# Tentukan pemenang berdasarkan F1-Score
if f1_tf >= f1_pt:
    WINNER = 'Deep MLP (TensorFlow)'
    best_model_save = model_tf
    best_f1 = f1_tf
else:
    WINNER = 'RNN (PyTorch)'
    best_model_save = model_pt
    best_f1 = f1_pt

print(f'\n>>> PEMENANG: {WINNER} (F1-Score = {best_f1:.4f})')

            Metrik  Deep MLP (TensorFlow)  RNN (PyTorch)
          Accuracy               0.926800       0.920400
         Precision               0.876372       0.875746
            Recall               0.886097       0.862598
          F1-Score               0.881207       0.869122
           AUC-ROC               0.976839       0.973139
Waktu Training (s)             143.100000      81.000000

>>> PEMENANG: Deep MLP (TensorFlow) (F1-Score = 0.8812)


In [91]:
# =============================================
# 11b. PLOT: Training History Kedua Model
# =============================================

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# TensorFlow loss
axes[0].plot(history_tf.history['loss'], label='Train Loss', color='steelblue')
axes[0].plot(history_tf.history['val_loss'], label='Val Loss', color='darkorange')
axes[0].set_title('Deep MLP — Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# TensorFlow accuracy
axes[1].plot(history_tf.history['accuracy'], label='Train Acc', color='steelblue')
axes[1].plot(history_tf.history['val_accuracy'], label='Val Acc', color='darkorange')
axes[1].set_title('Deep MLP — Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# PyTorch loss
axes[2].plot(history_pt['train_loss'], label='Train Loss', color='steelblue')
axes[2].plot(history_pt['val_loss'], label='Val Loss', color='darkorange')
axes[2].set_title('RNN (PyTorch) — Loss')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Loss')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Training History — Deep MLP vs RNN', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_history_comparison.png', dpi=150)
plt.show()

In [92]:
# =============================================
# 11c. PLOT: Confusion Matrix Kedua Model
# =============================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cm, title in [(axes[0], cm_tf, 'Deep MLP (TensorFlow)'),
                       (axes[1], cm_pt, 'RNN (PyTorch)')]:
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=[0,1], yticks=[0,1],
           xticklabels=['Not Compl.', 'Completed'],
           yticklabels=['Not Compl.', 'Completed'],
           title=title, ylabel='True', xlabel='Predicted')
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > thresh else 'black', fontsize=14)

plt.suptitle('Confusion Matrix Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix_comparison.png', dpi=150)
plt.show()

In [93]:
# =============================================
# 11d. PLOT: Bar Chart Perbandingan Metrik
# =============================================

metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'auc_roc']
vals_tf = [acc_tf, prec_tf, rec_tf, f1_tf, auc_tf]
vals_pt = [acc_pt, prec_pt, rec_pt, f1_pt, auc_pt]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, vals_tf, width, label='Deep MLP (TensorFlow)', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, vals_pt, width, label='RNN (PyTorch)', color='darkorange', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Score')
ax.set_title('Model Comparison — Deep MLP (TensorFlow) vs RNN (PyTorch)')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics])
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison_bar.png', dpi=150)
plt.show()

/tmp/ipykernel_58/2427565464.py:9: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(12, 6))


In [94]:
# =============================================
# 11e. PLOT: Radar Chart Perbandingan
# =============================================

angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]
vals_tf_r = vals_tf + vals_tf[:1]
vals_pt_r = vals_pt + vals_pt[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, vals_tf_r, 'o-', linewidth=2, label='Deep MLP (TF)', color='steelblue')
ax.fill(angles, vals_tf_r, alpha=0.15, color='steelblue')
ax.plot(angles, vals_pt_r, 'o-', linewidth=2, label='RNN (PyTorch)', color='darkorange')
ax.fill(angles, vals_pt_r, alpha=0.15, color='darkorange')

ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics], fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title('Model Performance Radar Chart', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/radar_chart_comparison.png', dpi=150)
plt.show()

## 12. Save Best Model + Full Pipeline (joblib)

In [95]:
# =============================================
# 12a. SIMPAN BEST MODEL (Keras .keras + PyTorch .pt)
# =============================================

# --- Simpan model TensorFlow ---
best_model_keras_path = f'{OUTPUT_DIR}/best_model.keras'
model_tf.save(best_model_keras_path)
print(f'Model TensorFlow disimpan: {best_model_keras_path}')

# --- Simpan model PyTorch ---
best_model_pt_path = f'{OUTPUT_DIR}/best_model_pytorch.pt'
torch.save({
    'model_state_dict': model_pt.state_dict(),
    'model_class': 'RNN',
    'input_dim': INPUT_DIM,
    'seq_len': SEQ_LEN,
    'feat_per_step': FEAT_PER_STEP,
}, best_model_pt_path)
print(f'Model PyTorch disimpan: {best_model_pt_path}')

Model TensorFlow disimpan: /kaggle/working/best_model.keras
Model PyTorch disimpan: /kaggle/working/best_model_pytorch.pt


In [96]:
# =============================================
# 12b. SIMPAN FULL PIPELINE BUNDLE (joblib)
# =============================================

pipeline_bundle = {
    # Preprocessing
    'preprocessor': preprocessor,
    'pca': pca,
    'lda': lda,
    'smote': smote,
    'best_dr': BEST_DR,
    'n_pca': n_pca,
    'ae_latent': AE_LATENT,

    # Data info
    'input_features': list(X.columns),
    'target_col': TARGET_COL,
    'num_cols': num_cols,
    'ordinal_cols': ordinal_cols,
    'nominal_cols': nominal_cols,
    'ordinal_map': ordinal_map,

    # Model results
    'best_model_name': WINNER,
    'results': {
        'Deep MLP (TensorFlow)': {
            'accuracy': float(acc_tf), 'precision': float(prec_tf),
            'recall': float(rec_tf), 'f1_score': float(f1_tf),
            'auc_roc': float(auc_tf), 'training_time_sec': float(tf_time),
        },
        'RNN (PyTorch)': {
            'accuracy': float(acc_pt), 'precision': float(prec_pt),
            'recall': float(rec_pt), 'f1_score': float(f1_pt),
            'auc_roc': float(auc_pt), 'training_time_sec': float(pt_time),
        },
    },

    # Metadata
    'metadata': {
        'seed': SEED,
        'n_total_rows': int(len(df)),
        'n_features_before_dr': int(n_features),
        'n_features_after_dr': int(INPUT_DIM),
        'dr_scores': {k: float(v) for k, v in dr_scores.items()},
        'train_size': int(X_train_res.shape[0]),
        'val_size': int(len(y_val)),
        'test_size': int(len(y_test)),
    },
}

bundle_path = f'{OUTPUT_DIR}/dl_pipeline_bundle.joblib'
joblib.dump(pipeline_bundle, bundle_path, compress=3)
print(f'Pipeline bundle disimpan: {bundle_path}')

Pipeline bundle disimpan: /kaggle/working/dl_pipeline_bundle.joblib


## 13. Inference Test — Prediksi Data Baru

In [97]:
# =============================================
# 13. INFERENCE TEST: Buktikan pipeline bisa predict data baru
# =============================================

sample_idx = np.random.choice(len(X_test_dr), size=10, replace=False)
X_sample_dr = X_test_dr[sample_idx]
y_true_s = y_test[sample_idx]

# --- Prediksi dengan Deep MLP (TensorFlow) ---
probs_tf_s = model_tf.predict(X_sample_dr, verbose=0).ravel()
preds_tf_s = (probs_tf_s >= 0.5).astype(int)

# --- Prediksi dengan RNN (PyTorch) ---
X_s_seq = torch.FloatTensor(reshape_to_seq(X_sample_dr, SEQ_LEN)).to(DEVICE)
with torch.no_grad():
    probs_pt_s = model_pt(X_s_seq).cpu().numpy().ravel()
preds_pt_s = (probs_pt_s >= 0.5).astype(int)

print(f'Demo Prediksi — 10 sampel acak dari test set')
print(f'{"#":<4} {"True":<6} {"TF Pred":<8} {"TF Prob":<10} {"PT Pred":<8} {"PT Prob":<10} {"Status"}')
print('-' * 65)
for i in range(10):
    tf_ok = 'OK' if preds_tf_s[i] == y_true_s[i] else 'WRONG'
    pt_ok = 'OK' if preds_pt_s[i] == y_true_s[i] else 'WRONG'
    print(f'{i+1:<4} {y_true_s[i]:<6} {preds_tf_s[i]:<8} {probs_tf_s[i]:<10.4f} {preds_pt_s[i]:<8} {probs_pt_s[i]:<10.4f} TF:{tf_ok} PT:{pt_ok}')

tf_demo_acc = np.mean(preds_tf_s == y_true_s)
pt_demo_acc = np.mean(preds_pt_s == y_true_s)
print(f'\nDemo Accuracy — TF: {tf_demo_acc*100:.0f}% | PT: {pt_demo_acc*100:.0f}%')

Demo Prediksi — 10 sampel acak dari test set
#    True   TF Pred  TF Prob    PT Pred  PT Prob    Status
-----------------------------------------------------------------
1    0      0        0.0330     0        0.2898     TF:OK PT:OK
2    0      1        0.5415     0        0.1913     TF:WRONG PT:OK
3    0      0        0.0008     0        0.0036     TF:OK PT:OK
4    1      1        0.9897     1        0.9981     TF:OK PT:OK
5    0      0        0.1801     0        0.1022     TF:OK PT:OK
6    0      0        0.1371     0        0.2496     TF:OK PT:OK
7    0      0        0.2366     0        0.1332     TF:OK PT:OK
8    1      0        0.1695     0        0.1347     TF:WRONG PT:WRONG
9    0      0        0.0545     0        0.0008     TF:OK PT:OK
10   0      0        0.0005     0        0.0005     TF:OK PT:OK

Demo Accuracy — TF: 80% | PT: 90%


## 14. Ringkasan & Kesimpulan

In [98]:
# =============================================
# 15. RINGKASAN PIPELINE
# =============================================

print('=' * 70)
print('  RINGKASAN END-TO-END DEEP LEARNING PIPELINE')
print('=' * 70)

print(f'''
  DATASET:
    - Sumber: digital_learning_analytics_100k.csv
    - Baris: {len(df):,} | Kolom: {df.shape[1]} (setelah cleaning)

  PREPROCESSING:
    - Drop kolom: ID, tanggal, data leakage ({len(DROP_COLS)} kolom)
    - Outlier: IQR Clipping
    - Encoding: Ordinal ({len(ordinal_cols)} kolom) + One-Hot ({len(nominal_cols)} kolom)
    - Scaling: StandardScaler
    - Fitur awal: {n_features} → Setelah DR: {INPUT_DIM} (via {BEST_DR})

  DIMENSIONALITY REDUCTION:
    - PCA: {n_pca} komponen (95% varians)
    - LDA: 1 komponen (class-separability)
    - PCA+LDA: {n_pca+1} komponen
    - Autoencoder: {AE_LATENT} latent dim
    - Terbaik: {BEST_DR} (F1 benchmark)

  MODEL:
    1. Deep MLP (TensorFlow)  — F1={f1_tf:.4f}, AUC={auc_tf:.4f}, Time={tf_time:.1f}s
    2. RNN (PyTorch)          — F1={f1_pt:.4f}, AUC={auc_pt:.4f}, Time={pt_time:.1f}s
    PEMENANG: {WINNER}

  FILES:
    - {best_model_keras_path}
    - {best_model_pt_path}
    - {bundle_path}
    - {OUTPUT_DIR}/training_history_comparison.png
    - {OUTPUT_DIR}/confusion_matrix_comparison.png
    - {OUTPUT_DIR}/model_comparison_bar.png
    - {OUTPUT_DIR}/radar_chart_comparison.png
    - {OUTPUT_DIR}/pca_variance_analysis.png
    - {OUTPUT_DIR}/dim_reduction_comparison.png
    - {OUTPUT_DIR}/huggingface_deploy_plan.txt
''')

  RINGKASAN END-TO-END DEEP LEARNING PIPELINE

  DATASET:
    - Sumber: digital_learning_analytics_100k.csv
    - Baris: 100,000 | Kolom: 38 (setelah cleaning)

  PREPROCESSING:
    - Drop kolom: ID, tanggal, data leakage (5 kolom)
    - Outlier: IQR Clipping
    - Encoding: Ordinal (2 kolom) + One-Hot (7 kolom)
    - Scaling: StandardScaler
    - Fitur awal: 116 → Setelah DR: 50 (via PCA+LDA)

  DIMENSIONALITY REDUCTION:
    - PCA: 49 komponen (95% varians)
    - LDA: 1 komponen (class-separability)
    - PCA+LDA: 50 komponen
    - Autoencoder: 25 latent dim
    - Terbaik: PCA+LDA (F1 benchmark)

  MODEL:
    1. Deep MLP (TensorFlow)  — F1=0.8812, AUC=0.9768, Time=143.1s
    2. RNN (PyTorch)          — F1=0.8691, AUC=0.9731, Time=81.0s
    PEMENANG: Deep MLP (TensorFlow)

  FILES:
    - /kaggle/working/best_model.keras
    - /kaggle/working/best_model_pytorch.pt
    - /kaggle/working/dl_pipeline_bundle.joblib
    - /kaggle/working/training_history_comparison.png
    - /kaggle/working/